# Learn Modern LangChain

This notebook teaches the latest LangChain patterns for building applications with language models. It covers:

- Installation and environment setup
- Chat models and prompt templates
- Chains and composability
- Conversation memory
- Document retrieval and QA
- Agents and tool use
- Advanced app-building ideas

## 1. Install dependencies

Install the core packages for LangChain app building. In a notebook or terminal, run:

```bash
pip install langchain openai python-dotenv faiss-cpu langchain_openai
```

If you use additional tools like SerpAPI or browser search, install them separately.

In [1]:
!pip install langchain openai python-dotenv faiss-cpu langchain_openai langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 2. Setup environment variables

LangChain uses model providers such as OpenAI. Set your API key in a `.env` file or environment variable.

```python
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
assert OPENAI_API_KEY, 'Set OPENAI_API_KEY in your environment.'
```

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os
from google.colab import userdata

# Get OPENAI_API_KEY from Google Colab secrets
# Ensure you have set an 'OPENAI_API_KEY' secret in your Colab secrets manager.
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('openapi')
    print("OPENAI_API_KEY successfully loaded from Colab secrets.")
except userdata.SecretNotFoundError:
    print("OPENAI_API_KEY not found in Colab secrets. Please set it in the Secrets panel.")
    # Optionally, you can set a placeholder or raise an error if the key is mandatory.
    os.environ['OPENAI_API_KEY'] = 'YOUR_OPENAI_API_KEY_PLACEHOLDER' # Fallback for clear error in subsequent cells

OPENAI_API_KEY successfully loaded from Colab secrets.


## 3. Chat models and prompt templates

Modern LangChain uses chat models and structured chat prompts. This is the foundation for dialog and application workflows.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0.7)
response = chat.invoke([
    SystemMessage(content='You are a helpful assistant.'),
    HumanMessage(content='Explain what LangChain is in one sentence.'),
])
response1 = chat.generate([
    [
        SystemMessage(content='You are a helpful assistant.'),
        HumanMessage(content='Explain what LangChain is in one sentence.'),
    ],
    [
        SystemMessage(content='You are a helpful assistant.'),
        HumanMessage(content='Explain what LangGraph is in one sentence.'),
    ],
    [
        SystemMessage(content='You are a helpful assistant.'),
        HumanMessage(content='what is today\'s climate. Give me output in json'),
    ],
    [
        SystemMessage(content='You are a helpful Diet assistant. Please provide details in detail in around 100 words'),
        HumanMessage(content='I am overweight, so how do i come to proper weight.'),
    ]
])
print(response.content)
print(response1.generations)

LangChain is a decentralized platform that uses blockchain technology to provide language learning services.
[[ChatGeneration(text='LangChain is a blockchain platform that focuses on multilingual content creation and translation services.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='LangChain is a blockchain platform that focuses on multilingual content creation and translation services.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 27, 'total_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DoiZ5y9tU1uWCC8HTHkTlwFdEe9Eq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_

In [5]:
print(response1.generations[0][0])
print('-------------------')
print(response1.generations[1][0])
print('-------------------')
print(response1.generations[2][0])
print('-------------------')
print(response1.generations[3][0].text)

text='LangChain is a blockchain platform that focuses on multilingual content creation and translation services.' generation_info={'finish_reason': 'stop', 'logprobs': None} message=AIMessage(content='LangChain is a blockchain platform that focuses on multilingual content creation and translation services.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 27, 'total_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DoiZ5y9tU1uWCC8HTHkTlwFdEe9Eq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eaaaf-5990-7fc2-ae7e-d149740056e2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_

In [6]:
response3 = chat.generate([
        [
        SystemMessage(content='You are a Indian Chef.'),
        HumanMessage(content='Make a eating stater in 25 minutes, give me step by step process'),
        ]
    ])
print(response3.generations[0][0].text)

Sure! Let's make a classic Indian appetizer called Vegetable Pakoras. Here's a step-by-step process to make them in 25 minutes:

Ingredients:
- 1 cup chickpea flour (besan)
- 1/4 cup rice flour
- 1 small onion, thinly sliced
- 1 small potato, thinly sliced
- 1 small carrot, grated
- 1/2 cup spinach, chopped
- 1 green chili, finely chopped
- 1/2 inch ginger, grated
- 1/2 teaspoon turmeric powder
- 1 teaspoon cumin seeds
- Salt to taste
- Water, as needed
- Oil for frying

Instructions:
1. In a mixing bowl, combine chickpea flour, rice flour, turmeric powder, cumin seeds, green chili, ginger, and salt. Mix well.
2. Gradually add water to the dry ingredients to make a thick batter. The batter should be thick enough to coat the vegetables.
3. Add the sliced onion, potato, grated carrot, and chopped spinach to the batter. Mix until all the vegetables are well coated.
4. Heat oil in a deep pan for frying. Make sure the oil is hot but not smoking.
5. Using a spoon or your hands, drop small po

In [7]:
!pip install langchain_community

## 4. Prompt templates

Use prompt templates to separate text structure from your application data. This increases reuse and reduces prompt duplication.

In [9]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

system_temp = SystemMessagePromptTemplate.from_template('You are a helpful AI Indian Chef assistant, for a given food items please give me a 15 minutes receipe steps that we can make.')
human_temp = HumanMessagePromptTemplate.from_template('Give me a receipe which i can make with following {items} food items.')


prompt1 = ChatPromptTemplate.from_messages([system_temp, human_temp])
chain = prompt1 | chat | StrOutputParser()
result = chain.invoke({'items': 'aloo, mirchi'})
print(result)

Sure! You can make "Aloo Mirchi Sabzi" with the ingredients aloo (potatoes) and mirchi (green chilies). Here is a simple recipe for Aloo Mirchi Sabzi:

Ingredients:
- 2 medium-sized potatoes (aloo)
- 4-5 green chilies (mirchi)
- 1 tablespoon oil
- 1 teaspoon cumin seeds (jeera)
- 1/2 teaspoon turmeric powder (haldi)
- 1 teaspoon red chili powder (adjust to taste)
- Salt to taste
- Chopped coriander leaves for garnishing

Instructions:
1. Peel the potatoes and chop them into small cubes. Slit the green chilies lengthwise.
2. Heat oil in a pan on medium heat. Add cumin seeds and let them splutter.
3. Add the chopped potatoes and green chilies to the pan. Mix well.
4. Add turmeric powder, red chili powder, and salt to taste. Mix everything together.
5. Cover the pan with a lid and let the potatoes cook on low heat. Stir occasionally to prevent sticking.
6. Cook until the potatoes are soft and cooked through. If needed, sprinkle a little water to help the potatoes cook.
7. Once the potatoe

In [10]:
prompt1.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful AI Indian Chef assistant, for a given food items please give me a 15 minutes receipe steps that we can make.'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['items'], input_types={}, partial_variables={}, template='Give me a receipe which i can make with following {items} food items.'), additional_kwargs={})]

In [13]:
syst_promt = SystemMessagePromptTemplate.from_template('You are a helpful AI Python Tutor, for a given {topic} please give me content which covers about, how to use, examples and real world problems are solved with this, i need to use as context for tutor chatbot to use it as content to check and answer.')
human_promt1 = HumanMessagePromptTemplate.from_template('Please help me with the following {question}.')

chat_prompt = ChatPromptTemplate.from_messages([syst_promt, human_promt1])
chain = chat_prompt | chat | StrOutputParser()
resp = chain.invoke({{'topic': "Multi-threading and Multiprocessing"},{'question': "how to write multi-threading to sole parallel problem?"}})
print(resp)


TypeError: unhashable type: 'dict'

## 5. Chains and composability

Chains let you compose multiple LLM calls into a workflow. Use them to implement multi-step app logic.

In [ ]:
from langchain.chains import LLMChain, SimpleSequentialChain
from langchain.prompts import PromptTemplate

summary_prompt = PromptTemplate(
    input_variables=['text'],
    template='Summarize the following text in one sentence:
{text}',
)
tweet_prompt = PromptTemplate(
    input_variables=['summary'],
    template='Write a short social media post based on this summary:
{summary}',
)

summary_chain = LLMChain(llm=chat, prompt=summary_prompt, output_key='summary')
tweet_chain = LLMChain(llm=chat, prompt=tweet_prompt)

overall_chain = SimpleSequentialChain(chains=[summary_chain, tweet_chain], verbose=True)
output = overall_chain.run('LangChain is a library that helps developers build applications with language models using chains, prompts, memory, and tools.')
print(output)

## 6. Conversation memory

Memory preserves chat state so the application can maintain context across turns.

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

conversation_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template('You are a helpful assistant that remembers what the user said earlier.'),
    HumanMessagePromptTemplate.from_template('{chat_history}
Human: {question}
Assistant:'),
])

conversation_chain = LLMChain(llm=chat, prompt=conversation_prompt, memory=memory, verbose=False)

print(conversation_chain.run({'question': 'What is LangChain?'}))
print(conversation_chain.run({'question': 'Now explain it again as if I were a beginner.'}))

## 7. Document retrieval and vector stores

Build a retriever that searches documents based on semantic embeddings. This is the basis for retrieval-augmented generation (RAG).

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

texts = [
    'LangChain is a framework for building applications with large language models.',
    'Retrieval augmentation improves accuracy by grounding answers in documents.',
    'Agents allow LangChain apps to call external tools and APIs.',
]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.create_documents(texts)

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
results = retriever.get_relevant_documents('How does LangChain use documents?')
for doc in results:
    print('---')
    print(doc.page_content)

## 8. Retrieval QA

Use the retriever with an LLM to answer questions from your document collection.

In [ ]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=chat,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True,
)

question = 'What does LangChain use to answer questions from documents?'
result = qa({'query': question})
print('Answer:')
print(result['result'])
print('
Sources:')
for doc in result['source_documents']:
    print('-', doc.page_content)

## 9. Agents and tools

Agents allow your app to call tools, run code, or fetch external data. This is powerful for building interactive applications.

In [ ]:
from langchain.agents import initialize_agent, Tool
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

def get_current_time(query: str) -> str:
    from datetime import datetime
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

time_tool = Tool(
    name='current_time',
    func=get_current_time,
    description='Returns the current date and time when asked.',
)

streaming_callbacks = CallbackManager([StreamingStdOutCallbackHandler()])
agent = initialize_agent(
    tools=[time_tool],
    llm=ChatOpenAI(model_name='gpt-3.5-turbo', streaming=True, callback_manager=streaming_callbacks),
    agent='zero-shot-react-description',
    verbose=True,
)

print(agent.run('What time is it right now?'))

## 10. Advanced app-building topics

Modern LangChain applications often include:

- Streaming and callback handling for responsive UIs
- Structured prompts for reliable outputs
- Hybrid search combining keyword and vector search
- Tool-enabled agents with external APIs or local functions
- Document loaders for PDFs, HTML, and databases
- Deployment patterns using FastAPI, Streamlit, or Gradio

Explore these as the next step once you have the basics working.

## Next steps

1. Add real document loaders for PDF, website, and database content.
2. Build a retrieval QA app with a web interface.
3. Create an agent that uses external tools like search, calendars, or custom APIs.
4. Experiment with chain-of-thought prompts and model selection.